In [1]:
# Import packages and set seed 

from pathlib import Path
from copy import deepcopy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, BatchNorm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.sparse import issparse
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
# Load the concrete dataset

CSV_PATH = "Concrete_Data.csv"   
TARGET_COL = "Concrete compressive strength(MPa, megapascals) "
df = pd.read_csv(CSV_PATH)
assert TARGET_COL in df.columns, f"Target missing; check column names: {df.columns.tolist()}"
print("Dataset:", df.shape)

Dataset: (1030, 9)


In [3]:
def make_X(df, mode="A"):
    if mode == "A":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Blast Furnace Slag (component 2)(kg in a m^3 mixture)",
            "Fly Ash (component 3)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Coarse Aggregate  (component 6)(kg in a m^3 mixture)",
            "Fine Aggregate (component 7)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "B":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "C":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
        tmp = df[[c_cem,c_slag,c_fly,c_wat,c_sp,c_ca,c_fa,c_age]].copy()
        tmp["Binder"]    = tmp[c_cem] + tmp[c_slag] + tmp[c_fly]
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["Fluid"]     = tmp[c_wat] + tmp[c_sp]
        eps = 1e-9
        tmp["w_b"]   = tmp[c_wat] / (tmp["Binder"] + eps)
        tmp["agg_b"] = tmp["Aggregate"] / (tmp["Binder"] + eps)
        tmp["sp_b"]  = tmp[c_sp] / (tmp["Binder"] + eps)
        return tmp[["Binder","Aggregate","Fluid","w_b","agg_b","sp_b", c_age]].copy()

    if mode == "D":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"

        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
    
        return tmp[[c_cem, c_slag, c_fly, c_wat, c_sp, "Aggregate", c_age]].copy()


    if mode == "E":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
    
        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["SEM"] = tmp[c_slag] + tmp[c_fly]
    
        return tmp[[c_cem, c_wat, c_sp, "Aggregate", "SEM", c_age]].copy()

In [4]:
# Choose the original feature group and reported K
MODE = "D"                   
ORIGINAL_K = {"A": 3, "B": 2, "C": 3, "D": 5, "E": 5}
k = ORIGINAL_K[MODE]
X_df = make_X(df, mode=MODE)
y_raw = df[TARGET_COL].to_numpy(dtype=float).reshape(-1, 1)
N = len(df)

idx = np.random.RandomState(42).permutation(N)
n_train, n_val = int(0.70 * N), int(0.15 * N)
train_idx = idx[:n_train]
val_idx = idx[n_train:n_train+n_val]
test_idx = idx[n_train+n_val:]
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
print(f"Model {MODE}, K={k} | Train {len(train_idx)}, Val {len(val_idx)}, Test {len(test_idx)}")

Model D, K=5 | Train 721, Val 154, Test 155


In [5]:
# Fit preprocessing and target scaling on TRAINING ONLY

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
transformers = [("num", numeric_pipe, num_cols)]
if cat_cols:
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", build_ohe()),
    ])
    transformers.append(("cat", categorical_pipe, cat_cols))
preprocess = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

X_train = preprocess.fit_transform(X_df.iloc[train_idx])
X_val = preprocess.transform(X_df.iloc[val_idx])
X_test = preprocess.transform(X_df.iloc[test_idx])

def dense_float(x):
    return np.asarray(x.toarray() if issparse(x) else x, dtype=np.float32)

X_train, X_val, X_test = map(dense_float, [X_train, X_val, X_test])
y_scaler = StandardScaler().fit(y_raw[train_idx])  # DO NOT use validation/test targets to fit.
y_train = y_scaler.transform(y_raw[train_idx]).ravel().astype(np.float32)
y_val = y_scaler.transform(y_raw[val_idx]).ravel().astype(np.float32)
y_test = y_scaler.transform(y_raw[test_idx]).ravel().astype(np.float32)
print("Scaled features:", X_train.shape, X_val.shape, X_test.shape)

Scaled features: (721, 7) (154, 7) (155, 7)


In [6]:
# Training graph: TRAIN nodes/edges ONLY; undirected, no explicit self-loops

def undirected_without_self_loops(edge_index_np):
    rows, cols = edge_index_np
    keep = rows != cols
    edges = np.column_stack([rows[keep], cols[keep]])
    edges = np.concatenate([edges, edges[:, ::-1]], axis=0)
    edges = np.unique(edges, axis=0)
    return torch.tensor(edges.T.copy(), dtype=torch.long)

nref = len(X_train)
assert 0 < k < nref
nbrs = NearestNeighbors(n_neighbors=k+1, metric="euclidean").fit(X_train)

train_neighbors = nbrs.kneighbors(X_train, n_neighbors=k+1, return_distance=False)
rows = np.repeat(np.arange(nref), k+1)
cols = train_neighbors.reshape(-1)
edge_train = undirected_without_self_loops(np.vstack([rows, cols])).to(device)
x_train = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
train_graph = Data(x=x_train, edge_index=edge_train, y=y_train_t)
assert edge_train.numel() > 0 and int(edge_train.max()) < nref
print("Training graph:", train_graph)

def make_query_graph(X_query):
    X_query = dense_float(X_query)
    n_query = len(X_query)
    neighbors = nbrs.kneighbors(X_query, n_neighbors=k, return_distance=False)
    src = neighbors.reshape(-1)
    dst = nref + np.repeat(np.arange(n_query), k)
    query_edges = torch.tensor(np.vstack([src, dst]), dtype=torch.long, device=device)
    edges = torch.cat([edge_train, query_edges], dim=1)
    x = torch.cat([x_train, torch.tensor(X_query, dtype=torch.float32, device=device)], dim=0)
    assert int(edges[0].max()) < nref  
    assert int(edges.max()) < nref + n_query
    return x, edges, slice(nref, nref+n_query)

val_x, val_edges, val_slice = make_query_graph(X_val)
# TEST graph is built AFTER model selection in Step 8; test features never enter training/validation.
print("Validation queries:", len(X_val), "| Training reference graph edges:", edge_train.shape[1])

Training graph: Data(x=[721, 7], edge_index=[2, 4648], y=[721])
Validation queries: 154 | Training reference graph edges: 4648


In [7]:
# GraphSAGE architecture and hyperparameters as the original Model A code

class SAGENet(nn.Module):
    def __init__(self, in_channels, hidden=128, layers=3, dropout=0.25, aggr="mean"):
        super().__init__()
        assert layers >= 2
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden, aggr=aggr))
        self.bns.append(BatchNorm(hidden))
        for _ in range(layers - 2):
            self.convs.append(SAGEConv(hidden, hidden, aggr=aggr))
            self.bns.append(BatchNorm(hidden))
        self.out_lin = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out_lin(x).view(-1)

set_seed(42)
model = SAGENet(in_channels=X_train.shape[1], hidden=128, layers=3,
                dropout=0.25, aggr="mean").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
print("Number of GraphSAGE convolutional layers:", len(model.convs))

Number of GraphSAGE convolutional layers: 2


In [8]:
# Train ONLY on training graph; early stop ONLY on validation queries
# No test loss, test predictions, or test graph is computed during training.

checkpoint_dir = Path("inductive_results")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / f"Model_{MODE}_K{k}_inductive_best.pt"
best_val, bad, best_epoch, best_state = float("inf"), 0, None, None
patience, min_delta, max_epochs = 30, 1e-4, 1000
history = []
y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
y_std_mpa = float(y_scaler.scale_[0])

for epoch in range(1, max_epochs+1):
    model.train()
    optimizer.zero_grad()
    pred_train = model(train_graph.x, train_graph.edge_index)
    train_loss = criterion(pred_train, train_graph.y)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        p_tr = model(train_graph.x, train_graph.edge_index)
        tr_loss_eval = criterion(p_tr, train_graph.y).item()
        p_val = model(val_x, val_edges)[val_slice]
        val_loss = criterion(p_val, y_val_t).item()

    history.append({"epoch": epoch,
                    "train_rmse_MPa": np.sqrt(tr_loss_eval)*y_std_mpa,
                    "val_rmse_MPa": np.sqrt(val_loss)*y_std_mpa})
    if val_loss < best_val - min_delta:
        best_val, bad, best_epoch = val_loss, 0, epoch
        best_state = deepcopy(model.state_dict())
    else:
        bad += 1
    if epoch == 1 or epoch % 25 == 0 or bad == 0:
        print(f"Epoch {epoch:4d} | train RMSE {history[-1]['train_rmse_MPa']:.4f} MPa | val RMSE {history[-1]['val_rmse_MPa']:.4f} MPa")
    if bad >= patience:
        print(f"Early stop at {epoch}; best validation epoch {best_epoch}")
        break

assert best_state is not None
model.load_state_dict(best_state)
torch.save(best_state, checkpoint_path)
print("Best model saved:", checkpoint_path)

Epoch    1 | train RMSE 16.4294 MPa | val RMSE 14.7976 MPa
Epoch    2 | train RMSE 15.1935 MPa | val RMSE 13.5724 MPa
Epoch    3 | train RMSE 14.1470 MPa | val RMSE 12.5769 MPa
Epoch    4 | train RMSE 13.3396 MPa | val RMSE 11.8337 MPa
Epoch    5 | train RMSE 12.7210 MPa | val RMSE 11.3299 MPa
Epoch    6 | train RMSE 12.2489 MPa | val RMSE 10.9931 MPa
Epoch    7 | train RMSE 11.8816 MPa | val RMSE 10.7595 MPa
Epoch    8 | train RMSE 11.6276 MPa | val RMSE 10.6316 MPa
Epoch    9 | train RMSE 11.4482 MPa | val RMSE 10.5773 MPa
Epoch   10 | train RMSE 11.3453 MPa | val RMSE 10.5756 MPa
Epoch   15 | train RMSE 11.1707 MPa | val RMSE 10.5663 MPa
Epoch   16 | train RMSE 11.0154 MPa | val RMSE 10.4292 MPa
Epoch   17 | train RMSE 10.7834 MPa | val RMSE 10.2176 MPa
Epoch   18 | train RMSE 10.5102 MPa | val RMSE 9.9688 MPa
Epoch   19 | train RMSE 10.2212 MPa | val RMSE 9.7130 MPa
Epoch   20 | train RMSE 9.9128 MPa | val RMSE 9.4277 MPa
Epoch   21 | train RMSE 9.6091 MPa | val RMSE 9.1493 MPa
Epo

In [9]:
# Evaluate VAL and held-out TEST SEPARATELY, once after model selection
@torch.no_grad()
def predict_query(X_query):
    model.eval()
    qx, qe, qs = make_query_graph(X_query)
    y_pred_scaled = model(qx, qe)[qs].cpu().numpy().reshape(-1,1)
    return y_scaler.inverse_transform(y_pred_scaled).ravel()

@torch.no_grad()
def predict_train():
    model.eval()
    yp = model(train_graph.x, train_graph.edge_index).cpu().numpy().reshape(-1,1)
    return y_scaler.inverse_transform(yp).ravel()

def metrics(actual, prediction):
    actual = np.asarray(actual).ravel()
    prediction = np.asarray(prediction).ravel()
    return {"R2": r2_score(actual, prediction),
            "MAE_MPa": mean_absolute_error(actual, prediction),
            "RMSE_MPa": np.sqrt(mean_squared_error(actual, prediction)),
            "MAPE_pct": 100*np.mean(np.abs((actual-prediction)/np.maximum(np.abs(actual),1e-9)))}

actual_train = y_raw[train_idx].ravel()
actual_val = y_raw[val_idx].ravel()
actual_test = y_raw[test_idx].ravel()
pred_train_mpa = predict_train()
pred_val_mpa = predict_query(X_val)
# The ONLY use of test features/targets for model performance is below.
pred_test_mpa = predict_query(X_test)

metric_df = pd.DataFrame({
    "Train": metrics(actual_train, pred_train_mpa),
    "Validation": metrics(actual_val, pred_val_mpa),
    "Test": metrics(actual_test, pred_test_mpa)
}).T
print("Model", MODE, "K", k, "| best validation epoch", best_epoch)
print(metric_df.round(4).to_string())

Model D K 5 | best validation epoch 444
                R2  MAE_MPa  RMSE_MPa  MAPE_pct
Train       0.9472   2.9270    3.9380   10.3522
Validation  0.8949   3.9088    5.0498   13.6987
Test        0.8989   3.9448    4.9834   13.9130


In [10]:
import sys
import torch
import torch_geometric
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
PyTorch Geometric: 2.8.0.post1
scikit-learn: 1.5.1
